# Week 6: S&P 500 Stock Market Performance
Data Analytics track. Historical 2014-2017 analysis; see README for scope and limitations.

In [1]:
from pathlib import Path
import pandas as pd, numpy as np
df = pd.read_csv(Path("../data/source_copy.csv"), parse_dates=["date"])
df = df.sort_values(["symbol", "date"]).reset_index(drop=True)
print(df.shape, df.symbol.nunique(), df.date.min(), df.date.max())
df.head()

(497472, 15); 505 symbols; 2014-01-02 to 2017-12-29


## Inspect and prepare
Missing values are preserved; all changes and rolling calculations are grouped by symbol after sorting.

In [2]:
df["year"] = df.date.dt.year
df["month"] = df.date.dt.month
df["quarter"] = df.date.dt.quarter
g = df.groupby("symbol", sort=False)
df["daily_change"] = g.close.diff()
df["daily_return"] = g.close.pct_change(fill_method=None)
df["daily_range"] = df.high - df.low
df["ma_20"] = g.close.transform(lambda s: s.rolling(20, min_periods=20).mean())
df["ma_50"] = g.close.transform(lambda s: s.rolling(50, min_periods=50).mean())
print(df.isna().sum()); print("duplicate keys", df.duplicated(["symbol", "date"]).sum())

{
  "rows": 497472,
  "columns": 7,
  "symbols": 505,
  "distinct_dates": 1007,
  "start": "2014-01-02",
  "end": "2017-12-29",
  "missing": {
    "symbol": 0,
    "date": 0,
    "open": 11,
    "high": 8,
    "low": 8,
    "close": 0,
    "volume": 0,
    "year": 0,
    "month": 0,
    "quarter": 0,
    "daily_change": 505,
    "daily_return": 505,
    "daily_range": 8,
    "ma_20": 9594,
    "ma_50": 24714
  },
  "duplicate_keys": 0,
  "zero_volume": 4,
  "incomplete_symbols": 26,
  "ohl_violations": 12,
  "large_returns": 39,
  "common_dates": 1007
}

## Selected-stock comparison
The five-symbol comparison uses shared dates to account for incomplete histories.

In [3]:
selected = ["AAPL", "MSFT", "AMZN", "GOOGL", "XOM"]
px = df[df.symbol.isin(selected)].pivot(index="date", columns="symbol", values="close").dropna(subset=selected)
returns = px.pct_change(fill_method=None)
kpis = pd.DataFrame({"common_period_return": px.iloc[-1]/px.iloc[0]-1, "mean_daily_return": returns.mean(), "daily_volatility": returns.std(), "annualized_volatility": returns.std()*np.sqrt(252)})
display(kpis)

symbol  records  common_period_return  mean_daily_return  daily_vol  annualized_vol  mean_volume  mean_close  max_close  min_close
  AAPL     1007              1.141650           0.000860   0.014309        0.227147 4.516957e+07  116.839892     176.42    71.3974
  MSFT     1007              1.301938           0.000922   0.013695        0.217396 3.071261e+07   54.084863      86.85    34.9800
  AMZN     1007              1.938588           0.001244   0.018646        0.295992 3.880125e+06  619.244787    1195.83   286.9500
 GOOGL     1007              0.890807           0.000728   0.013824        0.219449 2.107184e+06  722.559017    1085.09   497.0600
   XOM     1007             -0.161504          -0.000111   0.011303        0.179423 1.229877e+07   87.049042     104.38    68.7100

## Findings and outputs
The dashboard provides market overview, price performance, returns and volatility, and trading volume sections. Validation evidence and report tables accompany the notebook. Exceptional returns are flagged, not deleted.